# Check Models #

This Jupyter Notebook file can be used to check whether models are mass and charge balanced. Use checkAllModels() for multiple models or checkModel() for single model.

In [ ]:
import cobra
import cameo
import math
import escher
import plotly

import numpy as np
from scipy import stats
import pandas as pd
import sympy as sy
#from datetime import date, datetime
import time
import glob
import os

#date = datetime.strftime(datetime.now(), '%Y-%m-%d')

In [ ]:
## Main 1 - Checking multiple models##
#Run other functions below this cell first

#To check multiple models:
#inputDirectory_createModels = 'SP_Investigation/Overall_Analysis_Docs/RunModels/Input/XMLfiles/'
#inputDirectory_createModels = 'SP_Investigation/Product_Models/XML_Sartaaj_October-2020/'
inputDirectory_createModels = os.path.abspath("CreateProductModels/Output/")

tic = time.perf_counter() #Timer start
checkAllModels(inputDirectory_createModels)
toc = time.perf_counter()
print(f"Models checked in {toc - tic:0.4f} seconds")

In [ ]:
## Main 2 - Checking single model ##
#Run other functions below this cell first

#To check single model:
model = cameo.models.bigg.iAF692
#checkModel(model)
display(model.reactions.get_by_id('VOR'))
model.reactions.get_by_id('VOR').check_mass_balance()

In [ ]:
def checkAllModels(inputDirectory_createModels):

    for files in glob.glob(inputDirectory_createModels + '*.xml'):
        #Load model
        modelName = files.replace(inputDirectory_createModels, '') #Remove folder path from filenames
        model = cameo.load_model(inputDirectory_createModels + modelName)
        #display(modelName)#Check
        
        #Check the model
        check = checkModel(model)
        print('Model:', modelName)
        print('Is the model balanced: ', check[0])
        print('Unbalanced reactions:', check[1], '\n')
    
    print('Done evaluating models.')
    return()



In [ ]:
def checkModel(model):
#Function checks mass and charge balance of all reactions in the model
#Returns true or false based on whether model is mass balanced and a list of reactions that are not balanced (either in mass or charge), not including exchange reactions
    rxn_ids = [reaction.id for reaction in model.reactions]
    #display(rxn_ids) #Check
    
    unbalancedRxns = [] #Create empty list to store results
    for r in rxn_ids:
        balanceCheck = model.reactions.get_by_id(r).check_mass_balance()
        #print(r, 'check:', balanceCheck) #Check
        #If the list is not empty, the reaction is not mass balanced &/or not charge balanced.
        #All exchange reactions, DM reactions (sinks allowing certain compounds to leave)...
        #and BIOMASS rxns are not balanced, but are not included on "unbalanced" list
        if balanceCheck and not(r.startswith('EX_')) and not(r.startswith('DM_')) and not(r.startswith('BIOMASS')): 
            unbalancedRxns.append(r) #Unbalanced reaction IDs added to unbalancedRxns list
            #print(r, ' is not balanced.') #Check
        #else: #If list is empty, then the reaction is balanced both in mass and charge
            #print(r, ' is balanced.') #Check
    
    #Count unbalancedRxns to determine if model is balanced or not (true or false)
    if len(unbalancedRxns) is 0: #If no items in unbalancedRxns then model is balanced
        check = True
    else: #Else one or more items are in unbalancedRxns, model is not balanced
        check = False
    return(check, unbalancedRxns)